# Run federated g-shapcosim aggregation in Google Colab

This notebook prepares and launches the Flower server from this repository in Colab.

Steps:
- Provide a GitHub URL for the repository or upload the project files to Colab.
- Install dependencies (PyTorch, SHAP, Flower, W&B).
- (Optional) Mount Google Drive to persist SHAP outputs.
- Optionally set `aggregation-strategy = "g-shapcosim"` in `federated/pyproject.toml` (a cell below does this).

Run cells in order. The final cell runs `flwr run --stream` and will block the notebook (open a second tab for logs if needed).

In [ ]:
# 1) Clone or prepare the repo in Colab
REPO_URL = "https://github.com/b-fatma/master.git"
import os
import subprocess

WORKDIR = "/content/master"
if REPO_URL:
    print("Cloning repo:", REPO_URL)
    subprocess.run(["git", "clone", REPO_URL, WORKDIR], check=True)
else:
    print(
        "No REPO_URL set. Please upload your repo to /content/master or set REPO_URL and re-run this cell."
    )

os.makedirs(WORKDIR, exist_ok=True)
print("Working dir:", WORKDIR)

# 2) (Optional) Mount Google Drive to persist experiments (SHAP outputs, models, artifacts)
MOUNT_DRIVE = False  # set True to mount
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    # Example: link a folder for outputs
    os.makedirs('/content/drive/MyDrive/federated_runs', exist_ok=True)
    print('Drive mounted: /content/drive/MyDrive/federated_runs')
else:
    print('Drive mount skipped')

In [ ]:
# 3) Install dependencies (may take several minutes). Edit path if repo layout differs.
WORKDIR = "/content/master/feMMMMMMMMMMMMMMMMMMMMMNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNZ<<<<<<<<<<                                     NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN derated"
import os
import sys
import subprocess

os.chdir(WORKDIR)
print("Installing packages from requirements.txt (if present) and extras...")
req_file = os.path.join("..", "requirements.txt")
if os.path.exists(req_file):
    print("Installing project-level requirements...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", req_file], check=False
    )
else:
    print("No top-level requirements.txt found at", req_file)

# Install packages needed for SHAP + Flower + wandb
subprocess.run(
    [sys.executable, "-m", "pip", "install", "flwr", "shap", "wandb"], check=True
)

print("Installation complete. Verify torch is available (GPU recommended).")
import torch

print("torch:", torch.__version__, "cuda_available=", torch.cuda.is_available())

In [ ]:
# 4) Set W&B to offline mode to avoid login prompts and reduce network usage
import os

os.environ["WANDB_MODE"] = "offline"
print("WANDB_MODE=", os.environ.get("WANDB_MODE"))

In [ ]:
# 5) Optionally enable g-shapcosim aggregation and disable detection in the config
# This edits federated/pyproject.toml in-place. Backup first if desired.
cfg_path = "pyproject.toml"
print("Patching", cfg_path)
import re

with open(cfg_path, "r", encoding="utf-8") as f:
    text = f.read()

text = re.sub(
    r'aggregation-strategy\s*=\s*"[^"]+"', 'aggregation-strategy = "g-shapcosim"', text
)
# Optional: to avoid client exclusion, set detection-enabled = false
text = re.sub(
    r"detection-enabled\s*=\s*true",
    "detection-enabled = false",
    text,
    flags=re.IGNORECASE,
)

with open(cfg_path, "w", encoding="utf-8") as f:
    f.write(text)
print(
    "Patched pyproject.toml: aggregation-strategy=g-shapcosim, detection-enabled=false (if present)"
)

# Show patched lines for verification
for line in open(cfg_path, "r").read().splitlines():
    if "aggregation-strategy" in line or "detection-enabled" in line:
        print(line)

In [ ]:
# 6) Launch the Flower server (blocking cell). Open a second tab to monitor progress.
# Change to the federated directory and run Flower. This cell will block until the server stops.
import os
import subprocess
import sys

os.chdir("/content/master/federated")
print(
    "Starting Flower server in current notebook. This cell will block and stream logs."
)
print(
    "If you want to run in background, consider using screen/tmux on a VM instead of Colab."
)
env = os.environ.copy()
env["WANDB_MODE"] = env.get("WANDB_MODE", "offline")
try:
    subprocess.run(["flwr", "run", "--stream"], check=True, env=env)
except subprocess.CalledProcessError as e:
    print("Flower exited with", e)

Notes and troubleshooting:

- Colab runs have time and resource limits; long federated runs may be interrupted. For longer experiments use a VM or local machine.
- If SHAP imports fail, ensure `shap` is installed and restart the runtime. SHAP can be slow; use a GPU runtime and reduce `shap-background-size` in `pyproject.toml`.
- If you prefer to stream logs to a file instead of blocking the cell, replace the final `subprocess.run` with a background execution command and tail the logs.

If you want, I can also produce a minimal runnable example that simulates a few local clients in-process (no network) so you can validate the aggregation quickly in Colab.